In [2]:
# Load the support ticket classification dataset
df = pd.read_csv('dataset/support-ticket-classification.csv')
print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset loaded successfully!
Dataset shape: (800, 4)


,id,label,subject,text
0,1,account_access,"Password reset email not arriving, can't get b...",I clicked forgot password and submitted my ema...
1,2,account_access,Account locked after failed sign-in attempts,I tried to get into my account with the wrong ...
2,3,account_access,Two-factor authentication code not working,I set up 2FA but the app keeps generating inva...
3,4,account_access,Sign-in page showing blank white screen,The entry page loads but the form fields don't...
4,5,account_access,OAuth sign-in keeps failing,I tried to use Google to get into my account b...


In [3]:
# Explore dataset information
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("Dataset Statistics:")
print(df.describe())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       800 non-null    int64
 1   label    800 non-null    str  
 2   subject  800 non-null    str  
 3   text     800 non-null    str  
dtypes: int64(1), str(3)
memory usage: 25.1 KB
None

Missing Values:
id         0
label      0
subject    0
text       0
dtype: int64

Dataset Statistics:
              id
count  800.00000
mean   108.25000
std     77.61785
min      1.00000
25%     40.75000
50%     90.50000
75%    188.00000
max    275.00000


In [21]:
import re
print("✓ Basic imports ready (no external NLP libraries)")

✓ Basic imports ready (no external NLP libraries)


In [22]:
def manual_tokenize(text):
    """Tokenize by splitting on whitespace"""
    tokens = text.split()
    return [token for token in tokens if token]

def remove_punctuation(text):
    """Remove punctuation manually"""
    punctuation = '!\"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
    for p in punctuation:
        text = text.replace(p, '')
    return text

def simple_stemming(word):
    """Rule-based stemming without external libraries"""
    suffixes = [
        ('ies', 'i'), ('es', ''), ('s', ''), ('ed', ''), 
        ('ing', ''), ('ly', ''), ('er', ''), ('est', ''),
        ('tion', ''), ('sion', ''),
    ]
    
    word_lower = word.lower()
    for suffix, replacement in suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return word_lower[:-len(suffix)] + replacement
    return word_lower

def simple_lemmatization(word):
    """Dictionary-based lemmatization"""
    lemma_dict = {
        'was': 'be', 'were': 'be', 'been': 'be', 'being': 'be',
        'am': 'be', 'is': 'be', 'are': 'be',
        'have': 'have', 'has': 'have', 'had': 'have', 'having': 'have',
        'do': 'do', 'does': 'do', 'did': 'do', 'doing': 'do',
        'went': 'go', 'goes': 'go', 'going': 'go', 'gone': 'go',
        'running': 'run', 'runs': 'run', 'ran': 'run',
        'walked': 'walk', 'walks': 'walk', 'walking': 'walk',
        'tries': 'try', 'tried': 'try', 'trying': 'try',
    }
    word_lower = word.lower()
    return lemma_dict.get(word_lower, word_lower)

def handle_negations(text):
    """Replace negation contractions"""
    negations = {
        "n't": ' not', "cannot": ' not', "can't": ' not ',
        "won't": ' not', "didn't": ' not', "don't": ' not',
        "doesn't": ' not', "hadn't": ' not', "isn't": ' not',
        "aren't": ' not', "wasn't": ' not', "weren't": ' not',
        "hasn't": ' not', "haven't": ' not',
    }
    for neg, repl in negations.items():
        text = text.replace(neg, repl)
    return text

def custom_preprocess(text, subject):
    """Complete preprocessing pipeline from scratch"""
    # Concatenate
    combined = f"{subject} {text}"
    # Lowercase
    combined = combined.lower()
    # Handle negations
    combined = handle_negations(combined)
    # Remove punctuation
    combined = remove_punctuation(combined)
    # Normalize whitespace
    combined = ' '.join(combined.split())
    # Tokenize
    tokens = manual_tokenize(combined)
    # Lemmatize then stem
    processed = []
    for token in tokens:
        if len(token) > 0:
            lemma = simple_lemmatization(token)
            stem = simple_stemming(lemma)
            processed.append(stem)
    return ' '.join(processed)

print("✓ Custom preprocessing functions defined")

✓ Custom preprocessing functions defined


In [23]:
print("Applying custom preprocessing...")
df['processed_text'] = df.apply(lambda row: custom_preprocess(row['text'], row['subject']), axis=1)
print("✓ Preprocessing complete!\n")

print(f"Original: {df.iloc[0]['text'][:80]}...")
print(f"Processed: {df.iloc[0]['processed_text'][:80]}...")

Applying custom preprocessing...
✓ Preprocessing complete!

Original: I clicked forgot password and submitted my email address but I'm not receiving t...
Processed: password reset email not arriv ca not get back in i click forgot password and su...


In [24]:
import math

class CustomTFIDF:
    """TF-IDF implementation from scratch"""
    
    def __init__(self):
        self.vocabulary = {}
        self.idf_scores = {}
        self.vocab_size = 0
        
    def build_vocabulary(self, texts):
        """Build vocabulary from texts"""
        vocab = set()
        for text in texts:
            words = text.split()
            vocab.update(words)
        
        self.vocabulary = {word: idx for idx, word in enumerate(sorted(vocab))}
        self.vocab_size = len(self.vocabulary)
        print(f"✓ Vocabulary built: {self.vocab_size} unique terms")
        
    def calculate_idf(self, texts):
        """Calculate IDF scores for all terms"""
        num_docs = len(texts)
        doc_freq = {}
        
        # Count documents containing each term
        for text in texts:
            words = set(text.split())
            for word in words:
                doc_freq[word] = doc_freq.get(word, 0) + 1
        
        # Calculate IDF: log(total_docs / doc_freq)
        self.idf_scores = {}
        for word, freq in doc_freq.items():
            self.idf_scores[word] = math.log(num_docs / freq) if freq > 0 else 0
        
        print(f"✓ IDF scores calculated")
        
    def get_tfidf_vector(self, text):
        """Get TF-IDF vector for a single document"""
        vector = [0.0] * self.vocab_size
        
        # Calculate term frequencies
        words = text.split()
        word_freq = {}
        for word in words:
            word_freq[word] = word_freq.get(word, 0) + 1
        
        # Normalize TF by document length
        doc_length = len(words) if len(words) > 0 else 1
        
        # Calculate TF-IDF for each word in vocabulary
        for word, freq in word_freq.items():
            if word in self.vocabulary:
                idx = self.vocabulary[word]
                tf = freq / doc_length  # Normalized term frequency
                idf = self.idf_scores.get(word, 0)
                vector[idx] = tf * idf
        
        return vector
    
    def fit_transform(self, texts):
        """Fit the model and transform texts to TF-IDF vectors"""
        self.build_vocabulary(texts)
        self.calculate_idf(texts)
        
        # Transform all texts
        vectors = []
        for text in texts:
            vector = self.get_tfidf_vector(text)
            vectors.append(vector)
        
        print(f"✓ Transformed {len(vectors)} documents to TF-IDF vectors")
        return vectors

print("✓ CustomTFIDF class defined")

✓ CustomTFIDF class defined


In [25]:
print("Building TF-IDF model...\n")
tfidf = CustomTFIDF()
tfidf_vectors = tfidf.fit_transform(df['processed_text'].tolist())

print(f"\nTF-IDF Statistics:")
print(f"  - Vocabulary size: {tfidf.vocab_size}")
print(f"  - Number of documents: {len(tfidf_vectors)}")
print(f"  - Vector dimension: {len(tfidf_vectors[0])}")

# Show top terms by IDF score
sorted_idf = sorted(tfidf.idf_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 20 terms by IDF score:")
for i, (word, score) in enumerate(sorted_idf[:20], 1):
    print(f"  {i:2d}. {word:20s} - IDF: {score:.4f}")

Building TF-IDF model...

✓ Vocabulary built: 2044 unique terms
✓ IDF scores calculated
✓ Transformed 800 documents to TF-IDF vectors

TF-IDF Statistics:
  - Vocabulary size: 2044
  - Number of documents: 800
  - Vector dimension: 2044

Top 20 terms by IDF score:
   1. white                - IDF: 6.6846
   2. minuet               - IDF: 6.6846
   3. crea                 - IDF: 6.6846
   4. occurr               - IDF: 6.6846
   5. verfica              - IDF: 6.6846
   6. irreversible         - IDF: 6.6846
   7. deactivat            - IDF: 6.6846
   8. deactiva             - IDF: 6.6846
   9. verify               - IDF: 6.6846
  10. legacy               - IDF: 6.6846
  11. authenticator        - IDF: 6.6846
  12. somewhere            - IDF: 6.6846
  13. age                  - IDF: 6.6846
  14. government           - IDF: 6.6846
  15. corporate            - IDF: 6.6846
  16. profanity            - IDF: 6.6846
  17. nickname             - IDF: 6.6846
  18. inappropriate        - IDF: 6.684

In [26]:
import random

# Train/test split
random.seed(42)
indices = list(range(len(df)))
random.shuffle(indices)

split_ratio = 0.8
split_idx = int(len(df) * split_ratio)

train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

# Prepare training data
X_train = [tfidf_vectors[i] for i in train_indices]
y_train = [df.iloc[i]['label'] for i in train_indices]

# Prepare test data
X_test = [tfidf_vectors[i] for i in test_indices]
y_test = [df.iloc[i]['label'] for i in test_indices]

print(f"✓ Train/Test Split Complete")
print(f"  - Training set: {len(X_train)} samples")
print(f"  - Test set: {len(X_test)} samples")
print(f"  - Split ratio: {split_ratio*100:.0f}/{(1-split_ratio)*100:.0f}")
print(f"\nClass distribution in training set:")
train_classes = {}
for label in y_train:
    train_classes[label] = train_classes.get(label, 0) + 1
for label, count in sorted(train_classes.items()):
    print(f"  - {label}: {count}")

✓ Train/Test Split Complete
  - Training set: 640 samples
  - Test set: 160 samples
  - Split ratio: 80/20

Class distribution in training set:
  - account_access: 126
  - billing: 124
  - bug_report: 131
  - refund_request: 131
  - shipping_delivery: 128


In [29]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Training SVM classifier with sklearn...\n")
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train, y_train)

print("✓ SVM training complete")
print(f"\nMaking predictions on test set...")
y_pred = svm.predict(X_test)
print(f"✓ Predictions made on {len(y_pred)} test samples")

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*60}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"{'='*60}")

Training SVM classifier with sklearn...

✓ SVM training complete

Making predictions on test set...
✓ Predictions made on 160 test samples

Test Accuracy: 0.7188


In [30]:
print("DETAILED EVALUATION METRICS")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=sorted(set(y_test)))
classes_sorted = sorted(set(y_test))

print(f"{'Actual/Pred':<20}", end='')
for c in classes_sorted:
    print(f"{c:<15}", end='')
print()

for i, actual_class in enumerate(classes_sorted):
    print(f"{actual_class:<20}", end='')
    for j, pred_class in enumerate(classes_sorted):
        print(f"{cm[i][j]:<15}", end='')
    print()

DETAILED EVALUATION METRICS

Classification Report:
                   precision    recall  f1-score   support

   account_access       0.91      0.62      0.74        34
          billing       0.74      0.72      0.73        36
       bug_report       0.48      0.86      0.62        29
   refund_request       0.83      0.66      0.73        29
shipping_delivery       0.89      0.75      0.81        32

         accuracy                           0.72       160
        macro avg       0.77      0.72      0.73       160
     weighted avg       0.78      0.72      0.73       160


Confusion Matrix:
Actual/Pred         account_access billing        bug_report     refund_request shipping_delivery
account_access      21             1              12             0              0              
billing             0              26             6              3              1              
bug_report          2              1              25             0              1              
refund_re

In [35]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset

print("Loading DistilBERT (faster than BERT)...\n")
device = torch.device('cpu')

model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=5)
model.to(device)

# Label mapping
unique_labels = sorted(list(set(y_train)))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"✓ DistilBERT loaded")
print(f"  - Model: {model_name}")
print(f"  - Labels: {unique_labels}")

Loading DistilBERT (faster than BERT)...



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ DistilBERT loaded
  - Model: distilbert-base-uncased
  - Labels: ['account_access', 'billing', 'bug_report', 'refund_request', 'shipping_delivery']


In [36]:
print("Preparing data for DistilBERT...\n")

# Get original texts
X_train_texts = [df.iloc[i]['processed_text'] for i in train_indices]
X_test_texts = [df.iloc[i]['processed_text'] for i in test_indices]

# Tokenize
train_encodings = tokenizer(X_train_texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
train_labels = torch.tensor([label2id[label] for label in y_train])

test_encodings = tokenizer(X_test_texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
test_labels = torch.tensor([label2id[label] for label in y_test])

# Datasets
train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels)
test_dataset = TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], test_labels)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f"✓ Data prepared")
print(f"  - Training batches: {len(train_loader)}")
print(f"  - Test batches: {len(test_loader)}")

Preparing data for DistilBERT...

✓ Data prepared
  - Training batches: 40
  - Test batches: 10


In [37]:
print("Training DistilBERT...\n")

epochs = 2  # Fewer epochs for speed
learning_rate = 3e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (input_ids, attention_mask, labels) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(train_loader)
    train_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {train_acc:.4f}")

print("✓ Training complete")

Training DistilBERT...

Epoch 1/2 - Loss: 1.4020 - Accuracy: 0.3906
Epoch 2/2 - Loss: 0.7848 - Accuracy: 0.7844
✓ Training complete


In [38]:
print("Evaluating DistilBERT...\n")

from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert back to labels
y_pred_bert = [id2label[pred] for pred in all_preds]
y_test_labels = [id2label[label] for label in all_labels]

accuracy = sum(1 for i in range(len(y_test_labels)) if y_pred_bert[i] == y_test_labels[i]) / len(y_test_labels)

print(f"{'='*60}")
print(f"DistilBERT Test Accuracy: {accuracy:.4f}")
print(f"{'='*60}")
print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred_bert))

Evaluating DistilBERT...

DistilBERT Test Accuracy: 0.8187

Classification Report:
                   precision    recall  f1-score   support

   account_access       0.86      0.53      0.65        34
          billing       0.80      0.89      0.84        36
       bug_report       0.64      0.86      0.74        29
   refund_request       0.93      0.93      0.93        29
shipping_delivery       0.94      0.91      0.92        32

         accuracy                           0.82       160
        macro avg       0.83      0.82      0.82       160
     weighted avg       0.83      0.82      0.81       160

